In [1]:
# %%capture
# !pip install datasets==2.18.0
# !pip install transformers[torch]==4.38.2

In [2]:
%%capture
!pip install tensorboard
!pip install nltk
!pip install evaluate
!pip install transformers
!pip install datasets
!pip install sacrebleu accelerate

In [3]:
# %%capture
# # 1. Force remove existing conflicting versions
# !pip uninstall -y transformers accelerate datasets evaluate

# # 2. Install stable, modern versions that work together
# !pip install transformers==4.44.2 accelerate==0.33.0 datasets==2.21.0 evaluate sacrebleu

In [4]:
%%capture
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from transformers.integrations import TensorBoardCallback
from transformers import TrainerCallback
# Ensure the tokenizer data is downloaded (required in Kaggle/Colab)
nltk.download('punkt')
nltk.download('punkt_tab') # Required for newer NLTK versions
# Verify installation
import transformers
print(f"Transformers version: {transformers.__version__}")

2026-01-09 20:37:17.307336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767991037.485040      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767991037.537316      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767991037.961941      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767991037.961983      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767991037.961986      24 computation_placer.cc:177] computation placer alr

In [5]:
# %%capture
# import torch
# from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
# import pandas as pd
# from datasets import Dataset
# from transformers import AutoTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback, Seq2SeqTrainingArguments ,AutoModelForSeq2SeqLM
# import os
# from sklearn.model_selection import train_test_split
# import numpy as np
# from transformers.integrations import TensorBoardCallback
# from huggingface_hub import login
# import nltk
# import nltk
# from nltk.translate.bleu_score import sentence_bleu
# nltk.download("punkt")

## Load The model

In [6]:
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M", src_lang="ben_Beng" , tgt_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [7]:
article = "আমি আজ খুব খুশি"
inputs = tokenizer(article, return_tensors="pt")

In [8]:
# translated_tokens = model.generate(
#     **inputs, forced_bos_token_id=tokenizer.lang_code_to_id["arz_Arab"], 
#     max_length=360,
# )
# tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

translated_tokens = model.generate(
    **inputs, 
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng-Latn"), 
    max_length=360
)
tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

"I'm so happy today"

---

### freezing layers to reduce model size

In [9]:
# Freeze the encoder
for param in model.model.encoder.parameters():
    param.requires_grad = False

# for name, param in model.model.encoder.named_parameters():
#     if param.requires_grad:
#         print(f"{name} is trainable")
#     else:
#         print(f"{name} is frozen")


---

### Load Data

In [10]:
data_df = pd.read_csv("/kaggle/input/vashantor/all_dialects_train.csv")

In [11]:
test_df = pd.read_csv("/kaggle/input/vashantor/all_dialects_test.csv")

In [12]:
eval_df = pd.read_csv("/kaggle/input/vashantor/all_dialects_validation.csv")

---

## Training

In [13]:
tensorboard_callback = TensorBoardCallback()

In [14]:
max_length = 300

In [15]:
def tokenize_and_create_dataset(tokenizer, data_df, max_length=None):
    # Tokenize English text
    encodings = tokenizer(
        list(data_df["dialect_speech"]),
        truncation=True,
        padding=True,
        max_length=max_length  
    )

    # Tokenize Egyptian text
    decodings = tokenizer(
        list(data_df["english_speech"]),
        truncation=True,
        padding=True,
        max_length=max_length  
    )

    # Create Dataset object
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": decodings["input_ids"],
    })

    return dataset

In [16]:
# Function to tokenize data and create Dataset
def tokenize_and_create_dataset(tokenizer, data_df):
    encodings = tokenizer(list(data_df["dialect_speech"]), truncation=True, padding=True)
    decodings = tokenizer(list(data_df["english_speech"]), truncation=True, padding=True)

    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": decodings["input_ids"],
    })

    return dataset

train_df = data_df
# train_df, test_df = train_test_split(data_df, test_size=0.2, random_state=42)

train_dataset = tokenize_and_create_dataset(tokenizer, train_df)
test_dataset = tokenize_and_create_dataset(tokenizer, test_df)
eval_dataset = tokenize_and_create_dataset(tokenizer, eval_df)
print('Data Tokenized Successfully')

Data Tokenized Successfully


In [17]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [18]:
model_args = Seq2SeqTrainingArguments(
    output_dir="./output_dir",
    per_device_train_batch_size=8,       # Increased from 1
    gradient_accumulation_steps=4,       # Effective batch size = 32 (8x4)
    per_device_eval_batch_size=8,
    learning_rate=5e-5,                  # Slightly higher for faster convergence
    warmup_ratio=0.1,
    num_train_epochs=4,                  # Reduced from 6 to 3
    weight_decay=0.01,
    lr_scheduler_type="linear",
    eval_strategy="steps",         # Eval once per epoch instead of every 10k steps
    eval_steps=1000,
    save_steps=1000,
    logging_steps=200,
    save_strategy="epoch",
    metric_for_best_model="eval_loss",   # 🔴 REQUIRED
    greater_is_better=False,  
    predict_with_generate=True,
    fp16=True,                           # CRITICAL: Use Mixed Precision (if using NVIDIA GPU)
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=tokenizer,
    args=model_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # callbacks=[tensorboard_callback],
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1,   # stop after 1 bad epoch
            early_stopping_threshold=0.0
        )
    ],
)

trainer.train()


/tmp/ipykernel_24/2576986559.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.w

TrainOutput(global_step=588, training_loss=6.000269260536245, metrics={'train_runtime': 1788.9329, 'train_samples_per_second': 20.962, 'train_steps_per_second': 0.329, 'total_flos': 3809363558400000.0, 'train_loss': 6.000269260536245, 'epoch': 4.0})

In [19]:
torch.cuda.empty_cache()

## Eval 

In [20]:
import numpy as np

# 1. Preprocessing with -100 for padding labels
def preprocess_labels(labels_ids):
    # Replace tokenizer padding token id with -100
    return [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels_ids]

test_encodings = tokenizer(list(test_df["dialect_speech"]), truncation=True, padding=True)
test_labels = tokenizer(list(test_df["english_speech"]), truncation=True, padding=True)

test_dataset2 = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": preprocess_labels(test_labels["input_ids"]) # Use the fix here
})

# 2. Optimized Arguments
model_args = Seq2SeqTrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=16,      # Much faster than 1
    predict_with_generate=True,        # Necessary for real translation eval
    fp16=True,                         # Use if you have an NVIDIA GPU
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model, 
    args=model_args, 
    tokenizer=tokenizer
)

# 3. Perform evaluation
test_results2 = trainer.evaluate(test_dataset2)
print(test_results2)

/tmp/ipykernel_24/1629882517.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 1.3216809034347534, 'eval_runtime': 27.456, 'eval_samples_per_second': 68.291, 'eval_steps_per_second': 2.149}


In [21]:
translated_sentences_test = []
for sentence in test_df["dialect_speech"]:
    
    inputs = tokenizer(sentence, return_tensors="pt").to('cuda')
    
    translated_tokens = model.generate(
        **inputs, forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng-Latn"),
        max_length=360,
        no_repeat_ngram_size=3,
        num_return_sequences=1,
    ).to('cuda')
    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    translated_sentences_test.append(translated_text)

In [22]:
translated_sentences_test[:5]

['I came to Dhaka with my father from Sylhet',
 'Where is the fault of these three years?',
 'My daughter will never cry',
 'Nice to meet your friend',
 'Two days of our fair is very cold']

In [23]:
# Add the translated_text column to the evaluation dataset
test_df["translated_text"] = translated_sentences_test

In [24]:

english_texts = test_df['english_speech'].tolist()
translated_texts = test_df['translated_text'].tolist()

english_tokenized = [nltk.word_tokenize(text.lower()) for text in english_texts]
translated_tokenized = [nltk.word_tokenize(text.lower()) for text in translated_texts]

# Optional but HIGHLY recommended for sentence BLEU
smooth = SmoothingFunction().method1

bleu_scores = [
    sentence_bleu(
        [ref],          # reference must be a list of token lists
        hyp,            # hypothesis tokens
        smoothing_function=smooth
    )
    for ref, hyp in zip(english_tokenized, translated_tokenized)
]

avg_bleu_score = sum(bleu_scores) / len(bleu_scores)

print(f"Average BLEU Score after fine tuning: {avg_bleu_score}\n")

bleu_scores_df = pd.DataFrame(bleu_scores, columns=["BLEU"])
print("BLEU Scores (first 10):")
print(bleu_scores_df.head(10))

print(f'''0.0 to 0.1: Very poor translation quality.
0.1 to 0.3: Poor to fair translation quality.
0.3 to 0.5: Moderate translation quality.
0.5 to 0.7: Good translation quality.
0.7 to 0.9: Very good translation quality.
0.9 to 1.0: Excellent translation quality. \n''')





Average BLEU Score after fine tuning: 0.20385659712683352

BLEU Scores (first 10):
       BLEU
0  0.360888
1  0.467138
2  0.102950
3  1.000000
4  0.048010
5  0.023980
6  0.634047
7  0.194872
8  1.000000
9  0.027776
0.0 to 0.1: Very poor translation quality.
0.1 to 0.3: Poor to fair translation quality.
0.3 to 0.5: Moderate translation quality.
0.5 to 0.7: Good translation quality.
0.7 to 0.9: Very good translation quality.
0.9 to 1.0: Excellent translation quality. 

